# Clean Eilers Performance

Runs the incremental Eilers cleaning job, upserts the configured silver table, and records the latest source watermark.

In [0]:
dbutils.widgets.text("config_path", "config/clean_eilers.yml")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"])

config_path = dbutils.widgets.get("config_path")
full_refresh = dbutils.widgets.get("full_refresh").lower() == "true"

config_path, full_refresh

In [0]:
from pathlib import Path
import sys

repo_root = Path.cwd()
clean_eilers_path = repo_root / "scripts" / "cleaning" / "clean_eilers.py"
while repo_root != repo_root.parent and not clean_eilers_path.exists():
    repo_root = repo_root.parent
    clean_eilers_path = repo_root / "scripts" / "cleaning" / "clean_eilers.py"

cleaning_scripts_dir = repo_root / "scripts" / "cleaning"
if str(cleaning_scripts_dir) not in sys.path:
    sys.path.insert(0, str(cleaning_scripts_dir))

resolved_config_path = Path(config_path)
if not resolved_config_path.is_absolute():
    resolved_config_path = repo_root / resolved_config_path

from clean_eilers import run_clean_eilers

resolved_config_path

In [0]:
result = run_clean_eilers(
    resolved_config_path,
    spark=spark,
    write=True,
    full_refresh=full_refresh,
)

print(f"Rows processed: {result['row_count']:,}")
print(f"Operation: {result['write_operation']}")
print(f"Output table: {result['output_table']}")
print(f"Watermark table: {result['watermark_table']}")
print(f"Previous watermark: {result['previous_watermark']}")
print(f"New watermark: {result['new_watermark']}")

In [0]:
if result["missing_summary_df"] is not None:
    display(result["missing_summary_df"].orderBy("missing_percent", ascending=False))

In [0]:
if result["ownership_counts_df"] is not None:
    display(result["ownership_counts_df"])